# Project 5: ReAct Coding Agent

**Goal:** Build a ReAct (Reasoning + Acting) agent that solves coding problems by
interleaving chain-of-thought reasoning with tool use — executing Python code,
reading files, and searching documentation in a structured
thought → action → observation loop.

**New concept:** The agent reasoning loop.

**Why this matters for LLM engineering:** Every production LLM system that goes
beyond single-turn chat — Cursor, Devin, Claude Code, GitHub Copilot Workspace —
is built on some variant of the agent loop. Understanding how to structure tool
definitions, manage context windows, handle errors gracefully, and decide when
to stop is core to building reliable AI systems. This project teaches those
mechanics from scratch.

## 0. What We're Building On

### Concepts you already have (from Projects 1–4)

**From GPT (Project 3):**
- **Autoregressive generation**: the model produces one token at a time, each
  conditioned on everything before it. The ReAct agent extends this: instead of
  generating freely, we structure generation into alternating *thought* and
  *action* segments, then inject external *observations* back into the context
  before the next generation step.
- **Context window as working memory**: in your GPT, the entire sequence of
  tokens was the model's memory. In the agent, the context window is also the
  working memory — but now it contains structured blocks (thoughts, actions,
  observations) rather than raw text. Managing this memory is a first-class
  engineering problem.

**From ViT (Project 4):**
- **Bidirectional attention over structured inputs**: ViT taught you that
  attention can operate over non-sequential tokens (image patches). The agent's
  context contains heterogeneous blocks — natural language reasoning, code
  snippets, execution outputs — and the LLM attends over all of them
  bidirectionally when deciding what to do next.

### What is new in this project

The **agent reasoning loop** is the new concept. Instead of a model generating
text end-to-end, we build a *control loop* around the model:

```
while not done:
    thought = llm.generate("Think about what to do next")
    action  = llm.generate("Choose a tool and arguments")
    observation = execute(action)          # run the tool
    context.append(thought, action, observation)
```

This is not a neural architecture — it is a *program* that orchestrates an LLM.
The engineering challenges are:
1. **Tool definitions**: How do you tell the model what tools exist and how to call them?
2. **Parsing**: How do you reliably extract structured actions from free-form text?
3. **Context management**: The context window is finite. What do you do when it fills up?
4. **Termination**: How does the agent know when it is done?
5. **Error recovery**: What happens when a tool call fails or the model produces malformed output?

These are the same problems you would face building any production agent system.

## 1. Setup and Imports

**Note on this project:** Unlike your previous projects, this notebook does *not*
train a model from scratch. Instead, you will use a pre-trained LLM via API to
power your agent. The learning is in building the agent framework — the tool
definitions, the parsing logic, the loop control, and the context management.

We use the Deepseek as the backing LLM. You will need an API key.

In [3]:
import os
import re
import json
import time
import textwrap
import subprocess
import tempfile
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

# -- API setup --
# Set your DeepSeek API key before running:
#   export DEEPSEEK_API_KEY="sk-..."
#
# Get your key at: https://platform.deepseek.com

from openai import OpenAI
from dotenv import load_dotenv
load_dotenv('.env')  # loads .env from the current directory

client = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
print("DeepSeek client initialized")

MODEL = "deepseek-chat"

DeepSeek client initialized


## 2. The ReAct Framework

### 2.1 What is ReAct?

ReAct (Yao et al., 2022) interleaves **Re**asoning and **Act**ing in a single
LLM generation loop. The key insight: letting the model "think out loud" before
acting improves both the quality of actions *and* the interpretability of the
agent's behavior.

The loop structure:

```
Thought 1: I need to understand the function signature first.
Action 1:  read_file("solution.py")
Observation 1: [file contents appear here]

Thought 2: The function takes a list and returns the max. I should handle empty lists.
Action 2:  run_python("def find_max(lst): ...")
Observation 2: [execution result]

Thought 3: The implementation works. Let me test edge cases.
Action 3:  run_python("assert find_max([1,2,3]) == 3")
Observation 3: [test results]

Thought 4: All tests pass. I will submit the solution.
Action 4:  finish(solution="def find_max(lst): ...")
Observation 4: [DONE]
```

> **Why ReAct over pure chain-of-thought?** Pure CoT can reason but cannot
> verify — the model might think through a coding problem correctly but has no
> way to run the code and check. Pure acting (tool use without reasoning) tends
> to be brittle — the model calls tools without planning. ReAct combines both:
> reason about what to do, do it, observe the result, reason about what happened.
> This is exactly how Claude Code, Cursor, and similar tools work internally.

### 2.2 Comparison to autoregressive generation

In your GPT project, generation was:
```python
for _ in range(max_tokens):
    next_token = model(context)
    context.append(next_token)
```

The agent loop is structurally similar but with *external interrupts*:
```python
for _ in range(max_steps):
    thought_and_action = llm(context)       # generate
    context.append(thought_and_action)       # extend context
    observation = environment(action)        # EXTERNAL STEP
    context.append(observation)              # inject result
```

The critical difference: after each generation, control returns to *your code*,
which executes the action in the real world and feeds the result back. You are
writing the outer loop that orchestrates the LLM — the LLM is a component
inside your program, not the whole program.

## 3. Tool Definitions

### What tools are and why they matter

A "tool" is a function the agent can call. Each tool has:
- A **name** (string identifier)
- A **description** (natural language — this is what the LLM reads to decide when to use it)
- A **parameter schema** (what arguments it takes, with types)
- An **implementation** (actual Python function that runs when called)

The description is critical. The LLM has never seen your tool before — it
decides whether and how to use it based entirely on the description you write.
Vague descriptions lead to wrong tool choices. Overly detailed descriptions waste
context tokens.

> **Why this matters in production:** When you define tools for Claude, GPT-4,
> or any other model, the tool descriptions consume context window tokens on
> every single API call. A system with 20 tools, each with a 200-token
> description, burns 4,000 tokens of context just on tool definitions — before
> any conversation happens. This is why production systems carefully curate tool
> descriptions and sometimes dynamically select which tools to include based on
> the current task.

### Task 3.1: Define the tool schemas

Define three tools for our coding agent:

1. **`run_python`** — Executes a Python code snippet and returns stdout/stderr.
   - Parameter: `code` (string) — the Python code to execute.
   - Returns: stdout and stderr from execution, or an error message.

2. **`read_file`** — Reads the contents of a file from the workspace.
   - Parameter: `filepath` (string) — path to the file to read.
   - Returns: file contents as a string, or an error message.

3. **`finish`** — Signals that the agent has completed its task.
   - Parameter: `answer` (string) — the final answer or solution.
   - Returns: confirmation message.

The Deepseek API expects tool definitions in a specific JSON schema format.
Study the structure below and fill in the descriptions. Good descriptions should:
- State what the tool does in one sentence
- Specify any constraints (e.g., timeout, max output length)
- Give an example of when to use it vs. the other tools

In [4]:
def get_tool_definitions() -> list[dict]:
    tools = [
        {
            "type": "function",
            "function": {
                "name": "run_python",
                "description": "Execute a Python code snippet in an isolated subprocess and return stdout and stderr. Use this to test code, run computations, and verify solutions. Has a 30-second timeout and 10,000 character output limit. Prefer this over reasoning about what code will do — always run it to confirm.",  # YOUR CODE HERE
                "parameters": {
                    "type": "object",
                    "properties": {
                        "code": {
                            "type": "string",
                            "description": "Python code to execute"
                        }
                    },
                    "required": ["code"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "read_file",
                "description": "Read the contents of a file from the local filesystem and return it as a string. Use this to inspect existing files before modifying or analyzing them. Returns an error message if the file does not exist.",  # YOUR CODE HERE
                "parameters": {
                    "type": "object",
                    "properties": {
                        "filepath": {
                            "type": "string",
                            "description": "Path to the file to read"
                        }
                    },
                    "required": ["filepath"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "finish",
                "description": "Signal that the task is complete and return the final answer or solution. Call this only when you have verified the solution is correct by running it. Do not call this until all tests pass.",  # YOUR CODE HERE
                "parameters": {
                    "type": "object",
                    "properties": {
                        "answer": {
                            "type": "string",
                            "description": "The final answer or completed solution"
                        }
                    },
                    "required": ["answer"]
                }
            }
        },
    ]
    return tools

In [5]:
# -- Sanity check: tool definitions --
_tools = get_tool_definitions()
assert len(_tools) == 3
for _t in _tools:
    _fn = _t["function"]
    assert _fn["description"] != "", f"Tool '{_fn['name']}' has empty description"
    assert len(_fn["description"]) > 20
    assert "parameters" in _fn
print("\u2713 All 3 tools defined with non-empty descriptions")
for _t in _tools:
    _fn = _t["function"]
    print(f"  {_fn['name']}: {_fn['description'][:80]}...")

✓ All 3 tools defined with non-empty descriptions
  run_python: Execute a Python code snippet in an isolated subprocess and return stdout and st...
  read_file: Read the contents of a file from the local filesystem and return it as a string....
  finish: Signal that the task is complete and return the final answer or solution. Call t...


## 4. Tool Implementations

Now you will implement the actual functions that run when the agent calls each tool.

### Task 4.1: Implement `run_python`

This is the most important and most dangerous tool. It executes arbitrary Python
code in a subprocess. Key design decisions:

- **Timeout**: Code execution must have a timeout. An infinite loop in the
  agent's generated code should not hang your notebook forever. Use 30 seconds.
- **Output capture**: Capture both stdout and stderr. The agent needs to see
  error messages to debug its own code.
- **Output truncation**: LLM context windows are finite. If the code prints
  10,000 lines, you cannot feed all of that back. Truncate to a reasonable limit
  (e.g., 10,000 characters) and indicate truncation occurred.
- **Security**: In production, you would run this in a sandboxed container. For this
  notebook, we run in a subprocess with a timeout — not production-safe, but
  sufficient for learning.

> **Why subprocesses instead of `exec()`?** Using `exec()` runs the agent's code
> in your notebook's Python process. If the agent writes `import sys; sys.exit()`,
> your notebook dies. A subprocess isolates failures. This is the same reason
> production code execution tools (like those in Claude Code) use containers.

The function should:
1. Write the code to a temporary `.py` file
2. Run it with `subprocess.run()`, capturing stdout and stderr
3. Apply timeout of `timeout` seconds
4. Truncate output if it exceeds `max_output_chars`
5. Return a formatted string with stdout, stderr, and/or error info

In [6]:
def run_python(code: str, timeout: int = 30, max_output_chars: int = 10000) -> str:
    """
    Execute Python code in a subprocess and return the output.

    Args:
        code: Python source code to execute
        timeout: Maximum execution time in seconds
        max_output_chars: Maximum characters to return (truncate beyond this)

    Returns:
        str: Combined stdout/stderr output, or error message.
             Format:
               On success: "STDOUT:\n{stdout}\nSTDERR:\n{stderr}"
               On timeout: "ERROR: Execution timed out after {timeout} seconds"
               On error:   "ERROR: {error_message}"

    Implementation steps:
        1. Create a temporary .py file using tempfile.NamedTemporaryFile
           (use suffix='.py', mode='w', delete=False)
        2. Write the code string to the file, then close it
        3. Use subprocess.run() with:
           - ["python3", tmpfile_path] as the command
           - capture_output=True, text=True, timeout=timeout
        4. Catch subprocess.TimeoutExpired -> return timeout error message
        5. Catch any other Exception -> return generic error message
        6. On success: combine stdout and stderr into the output string
        7. If combined output exceeds max_output_chars, truncate and append
           "\n... [OUTPUT TRUNCATED]"
        8. Clean up the temp file in a finally block using os.unlink()
    """
    # YOUR CODE HERE
    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.py',mode='w',delete=False) as f:
            f.write(code)
            tmp_path = f.name
        result = subprocess.run(
        ["python3", tmp_path],
            capture_output=True,   # captures stdout and stderr instead of printing them
            text=True,             # returns strings instead of bytes
            timeout=timeout        # raises subprocess.TimeoutExpired if it runs too long
        )
        output = f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        if len(output) > max_output_chars:
            output = output[:max_output_chars] + "\n... [OUTPUT TRUNCATED]"

        return output
    except subprocess.TimeoutExpired:
        return f"ERROR: Execution timed out after {timeout} seconds"
    except Exception as e:
        return f"ERROR: {e}"
    finally:
        if tmp_path is not None:
            os.unlink(tmp_path)
    

In [7]:
# -- Sanity check: run_python --
_out = run_python("print('hello world')")
assert "hello" in _out, f"Expected 'hello' in output, got: {_out}"
print("\u2713 Basic execution works")

_out = run_python("x = 1/0")
assert "ZeroDivisionError" in _out, f"Expected ZeroDivisionError, got: {_out}"
print("\u2713 Error capture works")

_out = run_python("import time; time.sleep(60)", timeout=2)
assert "timed out" in _out.lower() or "timeout" in _out.lower(), (
    f"Expected timeout message, got: {_out}"
)
print("\u2713 Timeout works")

_out = run_python("print('x' * 50000)", max_output_chars=100)
assert len(_out) < 500, f"Expected truncated output, got {len(_out)} chars"
assert "TRUNCATED" in _out.upper(), "Expected truncation notice"
print("\u2713 Output truncation works")

print("\n\u2713 All run_python checks passed")

✓ Basic execution works
✓ Error capture works
✓ Timeout works
✓ Output truncation works

✓ All run_python checks passed


### Task 4.2: Implement `read_file`

Simple file reading with error handling. The agent might request files that do not
exist — return a clear error message so it can adjust.

The function should:
1. Check if the file exists
2. Read and return its contents
3. Truncate if the file is very large (same `max_chars` limit)
4. Return a clear error message if the file does not exist

In [8]:
def read_file(filepath: str, max_chars: int = 10000) -> str:
    """
    Read a file and return its contents.

    Args:
        filepath: Path to the file to read
        max_chars: Maximum characters to return

    Returns:
        str: File contents, or error message if file not found / unreadable.
             On success: the file text (truncated if needed)
             On not found: "ERROR: File not found: {filepath}"
             On other error: "ERROR: Could not read file: {error_message}"
    """
    # YOUR CODE HERE
    try:
        with open(filepath,mode='r') as f:
            content = f.read()
        if len(content) > max_chars:
            content = content[:max_chars]+ "\n... [OUTPUT TRUNCATED]"
        return content
    except FileNotFoundError:
        return f"ERROR: File not found: {filepath}"
    except Exception as e:
        return f"Could not read file: {e}"

In [9]:
# -- Sanity check: read_file --
_test_path = "/tmp/_react_test_file.py"
with open(_test_path, "w") as f:
    f.write("# test file\nprint('hello')\n")

_out = read_file(_test_path)
assert "test file" in _out, f"Expected file contents, got: {_out}"
print("\u2713 File reading works")

_out = read_file("/tmp/nonexistent_file_12345.py")
assert "ERROR" in _out or "not found" in _out.lower(), (
    f"Expected error message for missing file, got: {_out}"
)
print("\u2713 Missing file error works")

os.unlink(_test_path)
print("\n\u2713 All read_file checks passed")

✓ File reading works
✓ Missing file error works

✓ All read_file checks passed


### Task 4.3: Build the tool dispatcher

The tool dispatcher is a function that takes a tool name and arguments (as a dict)
and routes to the correct implementation. This is the bridge between the LLM's
structured output and your actual tool functions.

> **Why a dispatcher?** In production agent systems, the dispatcher is where you
> add logging, rate limiting, permission checks, and retry logic. It is the single
> point of control between "the LLM wants to do X" and "X actually happens."

In [12]:
def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    """
    Route a tool call to the appropriate implementation.

    Args:
        tool_name: One of "run_python", "read_file", "finish"
        tool_input: Dict of arguments matching the tool input_schema

    Returns:
        str: The tool output string

    Raises:
        ValueError: If tool_name is not recognized

    Implementation:
        - If tool_name == "run_python": call run_python(tool_input["code"])
        - If tool_name == "read_file": call read_file(tool_input["filepath"])
        - If tool_name == "finish": return f"TASK COMPLETE: {tool_input['answer']}"
        - Otherwise: raise ValueError(f"Unknown tool: {tool_name}")
    """
    # YOUR CODE HERE
    if tool_name == 'run_python':
        return run_python(tool_input['code'])
    elif tool_name == 'read_file':
        return read_file(tool_input['filepath'])
    elif tool_name == 'finish':
        return f"TASK COMPLETE: {tool_input['answer']}"
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

In [13]:
# -- Sanity check: dispatch_tool --
_out = dispatch_tool("run_python", {"code": "print(2+2)"})
assert "4" in _out, f"Expected '4' in output, got: {_out}"

_out = dispatch_tool("finish", {"answer": "done"})
assert "done" in _out.lower() or "COMPLETE" in _out, f"Unexpected finish output: {_out}"

try:
    dispatch_tool("nonexistent_tool", {})
    assert False, "Should have raised ValueError"
except ValueError:
    pass

print("\u2713 Tool dispatcher works correctly")

✓ Tool dispatcher works correctly


## 5. The Agent Loop

This is the core of the project. You will build the control loop that orchestrates
the LLM — calling it to think and act, executing actions, feeding observations
back, and deciding when to stop.

### 5.1 Agent state

First, we define data structures to track the agent's state. The conversation
history is a list of messages in the Deepseek API format, and we track metadata
about each step for debugging.

> **Why explicit state tracking?** In production, you need to log, debug, and
> replay agent runs. Keeping structured records of each step (not just the raw
> API messages) lets you build dashboards, catch failure patterns, and improve
> your prompts. Claude Code, for example, logs every tool call and its result.

In [14]:
@dataclass
class AgentStep:
    """Record of a single thought-action-observation cycle."""
    step_number: int
    thought: Optional[str]          # The model reasoning (from text blocks)
    tool_name: Optional[str]        # Which tool was called
    tool_input: Optional[dict]      # Arguments passed to the tool
    observation: Optional[str]      # Result from tool execution
    raw_response: Optional[dict] = None  # Full API response for debugging


@dataclass
class AgentState:
    """Complete state of an agent run."""
    task: str                                   # The original task description
    messages: list = field(default_factory=list) # Anthropic API message history
    steps: list = field(default_factory=list)    # List of AgentStep records
    is_done: bool = False                        # Whether the agent has finished
    final_answer: Optional[str] = None           # The answer from the finish tool
    total_input_tokens: int = 0                  # Running token count
    total_output_tokens: int = 0

### 5.2 System prompt

The system prompt tells the LLM how to behave as an agent. This is where you
define the agent's approach to problem-solving, how it should use tools, and
what it should do when things go wrong.

> **Why the system prompt matters so much:** In a standard chatbot, a mediocre
> system prompt just makes conversation slightly worse. In an agent, a bad
> system prompt causes the agent to call wrong tools, get stuck in loops, or
> never terminate. The system prompt is the agent's "training data" for how to
> behave — you cannot fine-tune the model, so the prompt is all you have.

### Task 5.2: Write the system prompt

Write a system prompt that instructs the LLM to:
1. Think step by step before acting
2. Use `run_python` to test code, not just reason about it
3. Use `read_file` to examine existing files before modifying them
4. Use `finish` when the task is complete, providing the final solution
5. Handle errors by reading the error message and trying a different approach
6. Keep responses focused — do not write essays between actions

In [15]:
def get_system_prompt() -> str:
    """
    Returns the system prompt that instructs the LLM how to behave as a
    ReAct coding agent.

    The prompt should:
    - Define the agent role (coding assistant that solves problems step by step)
    - Instruct it to think before acting (ReAct pattern)
    - Describe when to use each tool (run_python, read_file, finish)
    - Tell it to test code by running it, not just writing it
    - Instruct it to handle errors by examining the error and retrying
    - Tell it to call finish() with the final answer when done

    Returns:
        str: The system prompt
    """
    # YOUR CODE HERE
    prompt = """You are a coding agent that solves programming problems step by step.

            For every task, think carefully before acting. Reason about what you need to do, then use the appropriate tool.

            Tools:
            - run_python: Execute Python code to test solutions and verify they work. Always run code to confirm it works — do not just reason about what it will do.
            - read_file: Read an existing file before modifying or analyzing it.
            - finish: Call this when the task is complete with the verified final answer. Do not call finish until all tests pass.

            Error handling:
            - If a tool returns an error or failure, read the error message carefully and retry with a different approach.
            - If code raises an exception, debug it by examining the traceback and fixing the issue.

            Keep your reasoning concise. Do not write long explanations between actions.
            """
    return prompt

In [16]:
# -- Sanity check: system prompt --
_prompt = get_system_prompt()
assert len(_prompt) > 100, "System prompt seems too short to be effective"
assert len(_prompt) < 5000, "System prompt is very long -- may waste context tokens"

_lower = _prompt.lower()
_has_think = any(w in _lower for w in ["think", "reason", "thought", "step by step"])
_has_tool = any(w in _lower for w in ["tool", "run_python", "read_file", "finish"])
_has_error = any(w in _lower for w in ["error", "fail", "retry", "debug"])
assert _has_think, "Prompt should mention thinking/reasoning"
assert _has_tool, "Prompt should mention the available tools"
assert _has_error, "Prompt should mention error handling"

print(f"\u2713 System prompt is {len(_prompt)} chars")
print(f"\u2713 Mentions reasoning: {_has_think}")
print(f"\u2713 Mentions tools: {_has_tool}")
print(f"\u2713 Mentions error handling: {_has_error}")

✓ System prompt is 948 chars
✓ Mentions reasoning: True
✓ Mentions tools: True
✓ Mentions error handling: True


### 5.3 The core agent loop

Now the main event: the loop that drives the agent. Here is the control flow:

```
1. Send the conversation history to the LLM (with tools defined)
2. Parse the response:
   a. Extract any text blocks (the agent's "thoughts")
   b. Extract any tool_use blocks (the agent's "actions")
3. If the response has a tool call:
   a. Execute the tool via dispatch_tool()
   b. Append the assistant's response to the message history
   c. Append a tool_result message with the observation
   d. If the tool was "finish", set is_done = True
4. If the response has no tool call (just text):
   a. This should not happen if the prompt is good, but handle gracefully
5. Repeat until is_done or max_steps reached
```

**Key OpenAI/DeepSeek API detail:** When the model uses a tool, the response
message has `tool_calls` set (a list). The message also has `content` (the
model's thought, or None). You must append the assistant message, then send
back a `tool` role message to continue the conversation. The message structure is:

```python
# Assistant thinks and calls a tool — this is one message object
# message.content = "I'll run this code to test..."
# message.tool_calls = [
#     ToolCall(id="call_123", function=Function(
#         name="run_python", arguments='{"code": "print(42)"}'))
# ]

# You append the message object directly, then send the tool result:
{"role": "tool", "tool_call_id": "call_123",
 "content": "STDOUT:\n42\nSTDERR:\n"}
```

### Task 5.3: Implement the agent loop

This is the most important function in the notebook. Take your time.

**Implementation steps (detailed pseudocode):**

```python
def run_agent(task, max_steps=10):
    state = AgentState(task=task)
    system_prompt = get_system_prompt()
    state.messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task}
    ]
    tools = get_tool_definitions()

    for step_num in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=4096,
            messages=state.messages,
            tools=tools
        )

        state.total_input_tokens += response.usage.prompt_tokens
        state.total_output_tokens += response.usage.completion_tokens

        message = response.choices[0].message
        thought = message.content  # str or None
        
        # Append the raw assistant message
        state.messages.append(message)

        if message.tool_calls:
            tool_call = message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)  # <-- JSON string!
            tool_use_id = tool_call.id

            observation = dispatch_tool(tool_name, tool_input)

            state.messages.append({
                "role": "tool",
                "tool_call_id": tool_use_id,
                "content": observation
            })

            step = AgentStep(step_num, thought, tool_name, tool_input, observation)
            state.steps.append(step)
            # ... print step, check for finish ...
            if tool_name == "finish":
                state.is_done = True
                state.final_answer = tool_input.get("answer", "")
                break
        else:
            # text-only response
            ...

    return state
```

Implement this function. The pseudocode above is nearly complete — you need
to translate it into working Python, paying attention to:
- The exact message format the OpenAI/DeepSeek API expects
- Error handling (what if the API call fails?)
- Clean step logging with print statements

In [17]:
def run_agent(task: str, max_steps: int = 10) -> AgentState:
    """
    Run the ReAct agent loop on a given task.

    Args:
        task: Natural language description of the coding task
        max_steps: Maximum number of thought-action-observation cycles

    Returns:
        AgentState: Complete record of the agent run, including all steps,
                    messages, token usage, and final answer.

    The loop should:
        1. Initialize AgentState with the task as first messages:
           - {"role": "system", "content": get_system_prompt()}
           - {"role": "user", "content": task}
        2. On each step:
           a. Call client.chat.completions.create(
                  model=MODEL, max_tokens=4096,
                  messages=state.messages, tools=get_tool_definitions()
              )
           b. Get message = response.choices[0].message
              - thought = message.content (str or None)
              - tool calls = message.tool_calls (list or None)
           c. Append message to state.messages directly
           d. If message.tool_calls is not None:
              - tool_call = message.tool_calls[0]
              - tool_name = tool_call.function.name
              - tool_input = json.loads(tool_call.function.arguments)
                NOTE: arguments is a JSON string, not a dict!
              - observation = dispatch_tool(tool_name, tool_input)
              - Append {"role": "tool", "tool_call_id": tool_call.id,
                        "content": observation}
           e. If tool_name == "finish", set is_done and break
        3. Track tokens: response.usage.prompt_tokens / .completion_tokens
        4. Print each step for visibility
        5. Return the final state
    """
    # YOUR CODE HERE
    state = AgentState(task=task)
    state.messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": task}
    ]
    tools = get_tool_definitions()

    for step_num in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=4096,
            messages=state.messages,
            tools=tools
        )

        state.total_input_tokens += response.usage.prompt_tokens
        state.total_output_tokens += response.usage.completion_tokens

        message = response.choices[0].message
        thought = message.content

        msg_dict = {"role": "assistant", "content": thought}
        if message.tool_calls:
            msg_dict["tool_calls"] = [tc.model_dump() for tc in message.tool_calls]
        state.messages.append(msg_dict)

        if message.tool_calls:
            tool_call = message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            tool_use_id = tool_call.id

            observation = dispatch_tool(tool_name, tool_input)

            state.messages.append({
                "role": "tool",
                "tool_call_id": tool_use_id,
                "content": observation
            })

            step = AgentStep(step_num, thought, tool_name, tool_input, observation)
            state.steps.append(step)

            print(f"Step {step_num}: {tool_name}")
            if thought:
                print(f"  Thought: {thought[:100]}")
            print(f"  Observation: {observation[:100]}")

            if tool_name == "finish":
                state.is_done = True
                state.final_answer = tool_input.get("answer", "")
                break
        else:
            print(f"Step {step_num}: text-only response")
            if thought:
                print(f"  {thought[:200]}")

    return state

In [18]:
# -- Sanity check: agent loop structure --
# We cannot test the full loop without API calls, but we can verify the
# helper components are wired correctly.

_state = AgentState(task="test task")
assert _state.task == "test task"
assert _state.messages == []
assert _state.steps == []
assert _state.is_done == False
assert _state.final_answer is None

_step = AgentStep(
    step_number=1,
    thought="I should test this",
    tool_name="run_python",
    tool_input={"code": "print(1)"},
    observation="STDOUT:\n1\nSTDERR:\n"
)
assert _step.step_number == 1
assert _step.tool_name == "run_python"

print("\u2713 AgentState and AgentStep data structures work")
print("\u2713 Full loop test requires API key -- run the test tasks in Section 6")

✓ AgentState and AgentStep data structures work
✓ Full loop test requires API key -- run the test tasks in Section 6


## 6. Testing the Agent

Now let's test the agent on progressively harder tasks. These tests verify that
your agent loop, tool definitions, and system prompt all work together.

### Test 1: Simple computation

A minimal test — the agent should solve this in 1-2 steps.

In [19]:
# Test 1: Simple computation
state_1 = run_agent(
    "What is the sum of the first 100 prime numbers? "
    "Use Python to compute this. Give me just the number."
)

print(f"\n{'='*50}")
print(f"Final answer: {state_1.final_answer}")
print(f"Steps taken: {len(state_1.steps)}")
print(f"Tokens used: {state_1.total_input_tokens} in, {state_1.total_output_tokens} out")

Step 1: run_python
  Thought: I'll compute the sum of the first 100 prime numbers using Python.
  Observation: STDOUT:
First 100 primes: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 7
Step 2: run_python
  Thought: Let me verify this result by double-checking with a different approach to ensure accuracy.
  Observation: STDOUT:
Sum of first 100 primes (using sympy): 24133
Number of primes found: 100
Last prime: 541

Ve
Step 3: run_python
  Thought: Let me also verify by computing the sum manually to be absolutely certain.
  Observation: STDOUT:
Length of primes list: 100
Sum: 24133
All numbers are prime: True

STDERR:

Step 4: finish
  Thought: Perfect! All verifications confirm that the sum of the first 100 prime numbers is 24133.
  Observation: TASK COMPLETE: 24133

Final answer: 24133
Steps taken: 4
Tokens used: 5845 in, 1017 out


### Test 2: Multi-step problem with debugging

This task is designed to require multiple steps. The agent should write code,
encounter an issue, debug it, and iterate.

In [20]:
# Test 2: Multi-step problem
state_2 = run_agent(
    "Write a Python function called `flatten_nested` that takes an arbitrarily "
    "nested list (e.g., [1, [2, [3, 4], 5], [6]]) and returns a flat list "
    "([1, 2, 3, 4, 5, 6]). The function should handle any depth of nesting. "
    "Test it on at least 3 different inputs including an empty list and a "
    "deeply nested list. Return the final working function."
)

print(f"\n{'='*50}")
print(f"Final answer preview: {state_2.final_answer[:300] if state_2.final_answer else 'None'}...")
print(f"Steps taken: {len(state_2.steps)}")
print(f"Tokens used: {state_2.total_input_tokens} in, {state_2.total_output_tokens} out")

Step 1: run_python
  Thought: I'll write the `flatten_nested` function and test it with various inputs. Let me start by implementi
  Observation: STDOUT:
Test 1: Simple nested list
Input: [1, [2, [3, 4], 5], [6]]
Output: [1, 2, 3, 4, 5, 6]

Test 
Step 2: run_python
  Thought: Great! The function works correctly. Let me also test edge cases like a list containing only nested 
  Observation: STDOUT:
Test 7: List containing only nested lists
Input: [[[1, 2]], [[3]], [[4, 5, 6]]]
Output: [1, 
Step 3: finish
  Thought: Perfect! The function handles all test cases correctly. Now let me provide the final working functio
  Observation: TASK COMPLETE: def flatten_nested(nested_list):
    """
    Flatten an arbitrarily nested list into 

Final answer preview: def flatten_nested(nested_list):
    """
    Flatten an arbitrarily nested list into a single flat list.
    
    This function recursively traverses a nested list structure and returns
    a flat list containing all elements in depth-first

### Test 3: File reading + code generation

This test creates a file and asks the agent to read and work with it.

In [21]:
# Create a test file for the agent to read
_test_data = """name,age,score
Alice,25,92
Bob,30,85
Charlie,22,97
Diana,28,88
Eve,35,91
"""
with open("/tmp/students.csv", "w") as f:
    f.write(_test_data)

state_3 = run_agent(
    "Read the file /tmp/students.csv and write a Python script that:\n"
    "1. Finds the student with the highest score\n"
    "2. Calculates the average age\n"
    "3. Returns both results\n"
    "Run the script to verify it works."
)

print(f"\n{'='*50}")
print(f"Final answer: {state_3.final_answer}")
print(f"Steps taken: {len(state_3.steps)}")

Step 1: read_file
  Thought: I'll start by reading the file to understand its structure.
  Observation: name,age,score
Alice,25,92
Bob,30,85
Charlie,22,97
Diana,28,88
Eve,35,91

Step 2: run_python
  Thought: Now I can see the CSV structure: it has headers "name", "age", and "score". I'll write a Python scri
  Observation: STDOUT:
Student with highest score: Charlie (Score: 97)
Average age: 28.00

Result tuple: ('Charlie'
Step 3: run_python
  Thought: Great! The script works correctly. Let me also create a more robust version that handles potential e
  Observation: STDOUT:
Student with highest score: Charlie (Score: 97)
Average age: 28.00

Return values: (Charlie,
Step 4: finish
  Thought: Perfect! The script works correctly. Here's the final solution:
  Observation: TASK COMPLETE: The Python script successfully analyzes the students.csv file:

1. **Student with hig

Final answer: The Python script successfully analyzes the students.csv file:

1. **Student with highest score**: Charlie 

## 7. Context Window Management

### The problem

Every step of the agent loop adds to the message history:
- The assistant's thought + action (often 200-500 tokens)
- The tool result / observation (potentially thousands of tokens if code output is long)

After 10 steps, your message history might be 5,000-10,000 tokens. After 30
steps, you could hit the model's context window limit. This is a real problem
in production agent systems.

> **Why this matters in production:** Claude Code, Cursor, and similar tools all
> implement context management strategies. Common approaches:
> - **Sliding window**: Keep only the last N steps
> - **Summarization**: Periodically summarize old steps into a compact summary
> - **Selective retention**: Keep the system prompt, first message, and last K
>   steps, dropping everything in the middle
> - **Hierarchical memory**: Store old steps in a retrieval system and fetch
>   relevant ones when needed

### Task 7.1: Implement context truncation

Implement a function that truncates the message history when it gets too long.
Strategy: keep the first message (the original task) and the last `keep_last`
exchanges, dropping the middle.

This is a simplified version of what production systems do. The key insight is
that the most recent steps are usually most relevant (the agent's current line
of investigation), and the original task must always be present so the agent
does not forget what it is doing.

**Message structure to understand:**
- Messages alternate: user -> assistant -> user -> assistant -> ...
- The first message is the task (role="user")
- Each agent step produces: assistant message + user message (tool_result)
- So "the last K steps" means the last 2K messages

Your function should:
1. Count total messages
2. If count <= `keep_first + keep_last * 2`, return messages unchanged
3. Otherwise, keep the first `keep_first` messages, then the last `keep_last * 2`
   messages, dropping the middle
4. Insert a summary message in the gap so the agent knows context was dropped

In [26]:
def truncate_context(
    messages: list[dict],
    keep_first: int = 1,
    keep_last_steps: int = 4
) -> list[dict]:
    """
    Truncate message history to fit in context window.

    Args:
        messages: Full message history (list of API message dicts)
        keep_first: Number of initial messages to always keep (the task)
        keep_last_steps: Number of recent agent steps to keep.
                         Each step = 2 messages (assistant + tool_result),
                         so this keeps keep_last_steps * 2 messages from the end.

    Returns:
        list[dict]: Truncated message history with a summary marker in the middle.

    If total messages <= keep_first + keep_last_steps * 2:
        return messages unchanged (no truncation needed).

    Otherwise, return:
        messages[:keep_first]
        + [{"role": "user", "content": "[Earlier steps omitted for brevity. "
            "Focus on the current state and the original task.]"}]
        + messages[-(keep_last_steps * 2):]
    """
    # YOUR CODE HERE
    if len(messages) <= keep_first + keep_last_steps*2:
        return messages
    else:
        return(
            messages[:keep_first]
            + [{"role": "user", "content": "[Earlier steps omitted for brevity. "
                "Focus on the current state and the original task.]"}]
            + messages[-(keep_last_steps * 2):]
        )

In [27]:
# -- Sanity check: context truncation --
_msgs = [{"role": "user", "content": "Original task"}]
for i in range(10):
    _msgs.append({"role": "assistant", "content": f"Step {i} thinking"})
    _msgs.append({"role": "user", "content": f"Step {i} result"})

assert len(_msgs) == 21

# Should truncate
_truncated = truncate_context(_msgs, keep_first=1, keep_last_steps=3)
assert _truncated[0]["content"] == "Original task", "First message must be the task"
assert "omitted" in _truncated[1]["content"].lower() or "earlier" in _truncated[1]["content"].lower(), (
    "Second message should be the truncation marker"
)
assert len(_truncated) < len(_msgs), f"Should be shorter: {len(_truncated)} vs {len(_msgs)}"
assert _truncated[-1]["content"] == "Step 9 result", "Last message should be most recent"
print(f"\u2713 Truncation: {len(_msgs)} -> {len(_truncated)} messages")

# Should NOT truncate short histories
_short = _msgs[:5]
_result = truncate_context(_short, keep_first=1, keep_last_steps=3)
assert len(_result) == len(_short), "Short histories should not be truncated"
print("\u2713 Short histories pass through unchanged")

✓ Truncation: 21 -> 8 messages
✓ Short histories pass through unchanged


### Task 7.2: Integrate context management into the agent loop

Now modify your `run_agent` function to use context truncation. Create an
enhanced version that calls `truncate_context` before each API call when the
message history exceeds a threshold.

> **Design decision:** We truncate *before* sending to the API, not after.
> This means the agent always sees a manageable context, but we keep the full
> history in `state.messages` for debugging. Some production systems do it the
> other way (truncate the stored history) — both approaches have tradeoffs.

In [28]:
def run_agent_with_context_management(
    task: str,
    max_steps: int = 15,
    truncation_threshold: int = 10,
    keep_last_steps: int = 4
) -> AgentState:
    """
    Enhanced agent loop with context window management.

    Same as run_agent, but before each API call:
    1. Check if len(state.messages) > truncation_threshold
    2. If so, create a truncated copy of messages for the API call
       (but keep full history in state.messages for debugging)
    3. Use the truncated messages for the API call

    Args:
        task: The coding task to solve
        max_steps: Maximum agent steps
        truncation_threshold: Message count threshold before truncating
        keep_last_steps: Number of recent steps to keep when truncating

    Returns:
        AgentState with full (untruncated) history and all step records
    """
    # YOUR CODE HERE
    state = AgentState(task=task)
    state.messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": task}
    ]
    tools = get_tool_definitions()

    for step_num in range(1, max_steps + 1):

        # truncate before API call, but keep full history in state.messages
        if len(state.messages) > truncation_threshold:
            messages_for_api = truncate_context(state.messages, keep_last_steps=keep_last_steps)
        else:
            messages_for_api = state.messages

        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=4096,
            messages=messages_for_api,  # <-- truncated copy, not state.messages
            tools=tools
        )

        state.total_input_tokens += response.usage.prompt_tokens
        state.total_output_tokens += response.usage.completion_tokens

        message = response.choices[0].message
        thought = message.content

        msg_dict = {"role": "assistant", "content": thought}
        if message.tool_calls:
            msg_dict["tool_calls"] = [tc.model_dump() for tc in message.tool_calls]
        state.messages.append(msg_dict)  # always append to full history

        if message.tool_calls:
            tool_call = message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            tool_use_id = tool_call.id

            observation = dispatch_tool(tool_name, tool_input)

            state.messages.append({
                "role": "tool",
                "tool_call_id": tool_use_id,
                "content": observation
            })

            step = AgentStep(step_num, thought, tool_name, tool_input, observation)
            state.steps.append(step)

            print(f"Step {step_num}: {tool_name}")
            if thought:
                print(f"  Thought: {thought[:100]}")
            print(f"  Observation: {observation[:100]}")

            if tool_name == "finish":
                state.is_done = True
                state.final_answer = tool_input.get("answer", "")
                break
        else:
            print(f"Step {step_num}: text-only response")
    return state

## 8. Error Recovery and Robustness

### The problem

In production, agents fail in predictable ways:
1. **Malformed tool calls**: The model generates invalid JSON or calls a
   non-existent tool
2. **Tool execution errors**: Code throws an exception, file does not exist
3. **Infinite loops**: The agent gets stuck repeating the same action
4. **Context exhaustion**: The conversation gets too long for the model

Your current implementation handles (2) via error messages in tool output and
(4) via context truncation. Now let us handle (3) — loop detection.

### Task 8.1: Implement loop detection

A simple but effective heuristic: if the agent calls the same tool with the same
arguments N times in a row, it is probably stuck. Detect this and inject a
"you seem stuck" message to break the loop.

The function should examine the last `window` steps and check if they all have
the same tool name and tool input.

In [29]:
def detect_loop(steps: list[AgentStep], window: int = 3) -> bool:
    """
    Detect if the agent is stuck in a loop.

    Args:
        steps: List of completed AgentStep records
        window: Number of recent steps to check for repetition

    Returns:
        bool: True if the last `window` steps all used the same tool with
              the same input (indicating a loop), False otherwise.

    Edge cases:
        - If len(steps) < window, return False (not enough data)
        - Compare both tool_name and tool_input for equality
    """
    # YOUR CODE HERE
    if len(steps) < window:
        return False
    else:
        stuck = True
        for i in range(window-1):
            if steps[-i-1].tool_input != steps[-i-2].tool_input or steps[-i-1].tool_name != steps[-i-2].tool_name:
                stuck = False
        return stuck

In [30]:
# -- Sanity check: loop detection --
_steps = [
    AgentStep(1, "thinking", "run_python", {"code": "print(1)"}, "1"),
    AgentStep(2, "thinking", "run_python", {"code": "print(1)"}, "1"),
    AgentStep(3, "thinking", "run_python", {"code": "print(1)"}, "1"),
]
assert detect_loop(_steps, window=3) == True, "Should detect 3 identical steps as a loop"

_steps2 = [
    AgentStep(1, "thinking", "run_python", {"code": "print(1)"}, "1"),
    AgentStep(2, "thinking", "run_python", {"code": "print(2)"}, "2"),
    AgentStep(3, "thinking", "run_python", {"code": "print(1)"}, "1"),
]
assert detect_loop(_steps2, window=3) == False, "Different inputs -> not a loop"

_steps3 = [AgentStep(1, "t", "run_python", {"code": "x"}, "y")]
assert detect_loop(_steps3, window=3) == False, "Too few steps -> not a loop"

print("\u2713 Loop detection works correctly")

✓ Loop detection works correctly


## 9. Production-Grade Agent

### Task 9.1: Build the final agent

Combine everything — context management, loop detection, error handling — into
a polished agent runner. This version should:

1. Use context truncation when the history gets long
2. Detect loops and inject a "try a different approach" nudge
3. Handle API errors gracefully (retry with exponential backoff)
4. Print a clean step-by-step log
5. Return comprehensive state for analysis

This is as close to a production agent loop as you can get in a notebook.

In [31]:
def run_production_agent(
    task: str,
    max_steps: int = 15,
    truncation_threshold: int = 10,
    keep_last_steps: int = 4,
    loop_window: int = 3,
    max_retries: int = 2
) -> AgentState:
    """
    Production-grade ReAct agent with all robustness features.

    Args:
        task: The coding task to solve
        max_steps: Maximum number of agent steps
        truncation_threshold: When to start truncating context
        keep_last_steps: Recent steps to keep during truncation
        loop_window: Steps to examine for loop detection
        max_retries: API call retries on failure

    Returns:
        AgentState with full history and metadata

    Implementation plan:
        1. Initialize state and tools (same as run_agent)
        2. Main loop (max_steps iterations):
           a. Check for loops -> if detected, inject nudge message
           b. Prepare messages (truncate if needed, but keep full history)
           c. Call API with retry logic:
              - Try up to max_retries + 1 times
              - On failure, wait 2^attempt seconds (exponential backoff)
              - If all retries fail, log error and break
           d. Parse response (same as run_agent)
           e. Execute tool and record step (same as run_agent)
           f. Check for finish -> break if done
        3. Print summary statistics
        4. Return state
    """
    # YOUR CODE HERE
    state = AgentState(task=task)
    state.messages = [
        {"role": "system", "content": get_system_prompt()},
        {"role": "user", "content": task}
    ]
    tools = get_tool_definitions()

    for step_num in range(1, max_steps + 1):

        # a. loop detection — inject a nudge if stuck
        if detect_loop(state.steps, window=loop_window):
            print(f"  ⚠️  Loop detected — injecting nudge")
            state.messages.append({
                "role": "user",
                "content": "You seem to be repeating the same action. Try a different approach."
            })

        # b. context truncation — truncated copy for API, full history in state
        if len(state.messages) > truncation_threshold:
            messages_for_api = truncate_context(state.messages, keep_last_steps=keep_last_steps)
        else:
            messages_for_api = state.messages

        # c. API call with exponential backoff retry
        response = None
        for attempt in range(max_retries + 1):
            try:
                response = client.chat.completions.create(
                    model=MODEL,
                    max_tokens=4096,
                    messages=messages_for_api,
                    tools=tools
                )
                break
            except Exception as e:
                if attempt < max_retries:
                    wait = 2 ** attempt
                    print(f"  API error (attempt {attempt+1}/{max_retries+1}): {e}. Retrying in {wait}s...")
                    time.sleep(wait)
                else:
                    print(f"  API error: all retries failed. Stopping.")
                    return state

        state.total_input_tokens += response.usage.prompt_tokens
        state.total_output_tokens += response.usage.completion_tokens

        # d. parse response
        message = response.choices[0].message
        thought = message.content

        msg_dict = {"role": "assistant", "content": thought}
        if message.tool_calls:
            msg_dict["tool_calls"] = [tc.model_dump() for tc in message.tool_calls]
        state.messages.append(msg_dict)

        # e. execute tool and record step
        if message.tool_calls:
            tool_call = message.tool_calls[0]
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            tool_use_id = tool_call.id

            observation = dispatch_tool(tool_name, tool_input)

            state.messages.append({
                "role": "tool",
                "tool_call_id": tool_use_id,
                "content": observation
            })

            step = AgentStep(step_num, thought, tool_name, tool_input, observation)
            state.steps.append(step)

            print(f"Step {step_num}: {tool_name}")
            if thought:
                print(f"  Thought: {thought[:100]}")
            print(f"  Observation: {observation[:100]}")

            # f. check for finish
            if tool_name == "finish":
                state.is_done = True
                state.final_answer = tool_input.get("answer", "")
                break
        else:
            print(f"Step {step_num}: text-only response")
            if thought:
                print(f"  {thought[:200]}")

    # 3. summary
    print(f"\n{'='*50}")
    print(f"Task completed: {state.is_done}")
    print(f"Steps taken:    {len(state.steps)}")
    print(f"Tokens used:    {state.total_input_tokens} in, {state.total_output_tokens} out")
    if state.final_answer:
        print(f"Final answer:   {state.final_answer[:200]}")

    return state

### Test the production agent on a harder problem

In [32]:
# Test: A problem that requires debugging and iteration
state_hard = run_production_agent(
    "Write a Python class called `LRUCache` that implements a Least Recently Used "
    "cache with the following interface:\n"
    "- `__init__(self, capacity: int)` -- create a cache with given capacity\n"
    "- `get(self, key: str) -> int | None` -- return the value if key exists (and "
    "mark as recently used), else None\n"
    "- `put(self, key: str, value: int)` -- insert or update. If at capacity, "
    "evict the least recently used item.\n\n"
    "Test thoroughly with at least 5 test cases including edge cases "
    "(capacity 1, overwriting keys, eviction order). Return the final "
    "working implementation.",
    max_steps=15
)

print(f"\n{'='*50}")
print(f"Completed: {state_hard.is_done}")
print(f"Steps: {len(state_hard.steps)}")
print(f"Tokens: {state_hard.total_input_tokens} in, {state_hard.total_output_tokens} out")
if state_hard.final_answer:
    print(f"\nFinal answer preview:\n{state_hard.final_answer[:500]}...")

Step 1: run_python
  Thought: I'll implement the LRUCache class and test it thoroughly. Let me start by writing the implementation
  Observation: STDOUT:

STDERR:

Step 2: run_python
  Thought: Now let me test this implementation with various test cases, including edge cases.
  Observation: STDOUT:
Test 1: Basic functionality

STDERR:
Traceback (most recent call last):
  File "/var/folders
Step 3: run_python
  Thought: I need to define the class in the same code block. Let me run the tests with the class definition in
  Observation: STDOUT:
Test 1: Basic functionality
get('a'): 1
get('b'): 2
get('c'): None

Test 2: Eviction when at
Step 4: run_python
  Thought: Great! All tests are passing. However, I realize there's an issue with the current implementation: u
  Observation: STDOUT:
Test 1: Basic functionality
get('a'): 1
get('b'): 2
get('c'): None

Test 2: Eviction when at
Step 5: run_python
  Thought: Perfect! All tests pass. Let me also add a test for updating an existing key's valu

## 10. Analysis and Ablations

### 10.1 Step efficiency analysis

Let us analyze how the agent uses its steps. This tells you about the quality of
your system prompt and tool definitions.

In [33]:
def analyze_agent_run(state: AgentState) -> dict:
    """Analyze an agent run and return statistics."""
    stats = {
        "total_steps": len(state.steps),
        "completed": state.is_done,
        "tool_usage": {},
        "avg_thought_length": 0,
        "avg_observation_length": 0,
        "total_input_tokens": state.total_input_tokens,
        "total_output_tokens": state.total_output_tokens,
        "errors_encountered": 0,
    }

    thought_lengths = []
    obs_lengths = []

    for step in state.steps:
        name = step.tool_name or "none"
        stats["tool_usage"][name] = stats["tool_usage"].get(name, 0) + 1

        if step.thought:
            thought_lengths.append(len(step.thought))
        if step.observation:
            obs_lengths.append(len(step.observation))
            if "ERROR" in step.observation or "Traceback" in step.observation:
                stats["errors_encountered"] += 1

    stats["avg_thought_length"] = sum(thought_lengths) / max(len(thought_lengths), 1)
    stats["avg_observation_length"] = sum(obs_lengths) / max(len(obs_lengths), 1)

    return stats


# Analyze all test runs
for name, state in [("Simple", state_1), ("Multi-step", state_2),
                     ("File reading", state_3), ("LRU Cache", state_hard)]:
    stats = analyze_agent_run(state)
    print(f"\n{chr(9472)*40}")
    print(f"  {name}")
    print(f"   Steps: {stats['total_steps']}  |  Completed: {stats['completed']}")
    print(f"   Tool usage: {stats['tool_usage']}")
    print(f"   Errors encountered: {stats['errors_encountered']}")
    print(f"   Avg thought length: {stats['avg_thought_length']:.0f} chars")
    print(f"   Tokens: {stats['total_input_tokens']} in, {stats['total_output_tokens']} out")


────────────────────────────────────────
  Simple
   Steps: 4  |  Completed: True
   Tool usage: {'run_python': 3, 'finish': 1}
   Errors encountered: 0
   Avg thought length: 79 chars
   Tokens: 5845 in, 1017 out

────────────────────────────────────────
  Multi-step
   Steps: 3  |  Completed: True
   Tool usage: {'run_python': 2, 'finish': 1}
   Errors encountered: 0
   Avg thought length: 132 chars
   Tokens: 4981 in, 1763 out

────────────────────────────────────────
  File reading
   Steps: 4  |  Completed: True
   Tool usage: {'read_file': 1, 'run_python': 2, 'finish': 1}
   Errors encountered: 0
   Avg thought length: 128 chars
   Tokens: 4645 in, 1094 out

────────────────────────────────────────
  LRU Cache
   Steps: 8  |  Completed: True
   Tool usage: {'run_python': 7, 'finish': 1}
   Errors encountered: 1
   Avg thought length: 242 chars
   Tokens: 26985 in, 6766 out


### 10.2 Ablation: System prompt quality

The system prompt has an enormous effect on agent behavior. Let us test a
deliberately bad prompt vs. your prompt.

**Hypothesis:** A vague system prompt ("You are a helpful assistant.") will cause
the agent to take more steps, make more errors, and possibly fail to complete
the task. A good prompt that explicitly instructs ReAct-style behavior will be
more efficient.

In [35]:
# Ablation: system prompt quality comparison
def get_bad_system_prompt() -> str:
    return "You are a helpful assistant."

_test_task = (
    "Write a Python function that checks if a string is a palindrome. "
    "Test it on 'racecar', 'hello', and ''. Return the function."
)

print("=== Testing with YOUR system prompt ===")
state_good_prompt = run_agent(_test_task, max_steps=10)
stats_good = analyze_agent_run(state_good_prompt)

print("\n\n=== Testing with BAD system prompt ===")
_bad_state = AgentState(task=_test_task)
_bad_state.messages = [
    {"role": "system", "content": get_bad_system_prompt()},
    {"role": "user", "content": _test_task}
]

for _step_num in range(1, 11):
    try:
        _resp = client.chat.completions.create(
            model=MODEL, max_tokens=4096,
            messages=_bad_state.messages,
            tools=get_tool_definitions(),
        )
        _bad_state.total_input_tokens += _resp.usage.prompt_tokens
        _bad_state.total_output_tokens += _resp.usage.completion_tokens

        _message = _resp.choices[0].message
        _thought = _message.content
        _tool_name = _tool_input = _tool_id = None

        if _message.tool_calls:
            _tc = _message.tool_calls[0]
            _tool_name = _tc.function.name
            _tool_input = json.loads(_tc.function.arguments)
            _tool_id = _tc.id

        _bad_state.messages.append(_message)

        if _tool_name:
            _obs = dispatch_tool(_tool_name, _tool_input)
            _bad_state.messages.append({
                "role": "tool",
                "tool_call_id": _tool_id,
                "content": _obs
            })
            _bad_state.steps.append(
                AgentStep(_step_num, _thought, _tool_name, _tool_input, _obs)
            )
            print(f"Step {_step_num}: {_tool_name}")
            if _tool_name == "finish":
                _bad_state.is_done = True
                _bad_state.final_answer = _tool_input.get("answer", "")
                break
        else:
            print(f"Step {_step_num}: text only")
    except Exception as e:
        print(f"Step {_step_num}: API error: {e}")
        break

stats_bad = analyze_agent_run(_bad_state)

print(f"\n{'='*50}")
print(f"{'Metric':<25} {'Good prompt':>12} {'Bad prompt':>12}")
print(f"{chr(9472)*50}")
print(f"{'Steps taken':<25} {stats_good['total_steps']:>12} {stats_bad['total_steps']:>12}")
print(f"{'Completed':<25} {str(stats_good['completed']):>12} {str(stats_bad['completed']):>12}")
print(f"{'Errors':<25} {stats_good['errors_encountered']:>12} {stats_bad['errors_encountered']:>12}")
print(f"{'Input tokens':<25} {stats_good['total_input_tokens']:>12} {stats_bad['total_input_tokens']:>12}")
print(f"{'Output tokens':<25} {stats_good['total_output_tokens']:>12} {stats_bad['total_output_tokens']:>12}")

=== Testing with YOUR system prompt ===
Step 1: run_python
  Thought: I'll write a Python function to check if a string is a palindrome, test it on the given examples, an
  Observation: STDOUT:
Test results:
is_palindrome('racecar') = True
is_palindrome('hello') = False
is_palindrome('
Step 2: run_python
  Thought: Good! The function works correctly. Now let me also test with some more edge cases and verify the fu
  Observation: STDOUT:
Edge case tests:
is_palindrome(' ') = True
is_palindrome('  ') = True
is_palindrome('a') = T
Step 3: run_python
  Thought: I notice that the function doesn't handle punctuation properly (the last test returned False when it
  Observation: STDOUT:

STDERR:
  File "/var/folders/ds/_jnmtffd60z_1pp9v59v1s0m0000gn/T/tmpyh70wum2.py", line 24
 
Step 4: run_python
  Thought: Let me fix the syntax error with the f-string:
  Observation: STDOUT:
Test results for improved function:
is_palindrome('racecar') = True
is_palindrome('hello') =
Step 5: finish
  Thought: 

### Observations

Fill in based on your results:

> **Steps taken:** The good prompt took ___ steps vs ___ for the bad prompt.
> This difference is because ___.

> **Token efficiency:** The good prompt used ___ total tokens vs ___.
> In a production system at scale, this difference matters because
> ___ (think about cost and latency per request).

> **Error rate:** The good prompt encountered ___ errors vs ___.
> The good prompt reduces errors by ___ (what specifically does your
> prompt do that prevents errors?).

> **Connection to production:** This is why companies like Anthropic invest
> heavily in system prompt engineering for agent products. Claude Code's system
> prompt, for example, ___ (hypothesize about what design choices it makes
> based on what you have learned).

### 10.3 Ablation: Tool description quality

**Hypothesis:** Vague tool descriptions will cause the agent to misuse tools
or fail to use the right tool for a task.

In [36]:
# Ablation: tool description quality
def get_bad_tool_definitions():
    import copy
    tools = copy.deepcopy(get_tool_definitions())
    for t in tools:
        t["function"]["description"] = t["function"]["name"]
    return tools

print("=== Testing with GOOD tool descriptions ===")
_task = "Read the file /tmp/students.csv and tell me the name of the oldest student."
state_good_tools = run_agent(_task, max_steps=8)

print("\n\n=== Testing with BAD tool descriptions ===")
_bad_tools_state = AgentState(task=_task)
_bad_tools_state.messages = [
    {"role": "system", "content": get_system_prompt()},
    {"role": "user", "content": _task}
]

for _step_num in range(1, 9):
    try:
        _resp = client.chat.completions.create(
            model=MODEL, max_tokens=4096,
            messages=_bad_tools_state.messages,
            tools=get_bad_tool_definitions(),
        )
        _bad_tools_state.total_input_tokens += _resp.usage.prompt_tokens
        _bad_tools_state.total_output_tokens += _resp.usage.completion_tokens

        _message = _resp.choices[0].message
        _thought = _message.content
        _tool_name = _tool_input = _tool_id = None

        if _message.tool_calls:
            _tc = _message.tool_calls[0]
            _tool_name = _tc.function.name
            _tool_input = json.loads(_tc.function.arguments)
            _tool_id = _tc.id

        _bad_tools_state.messages.append(_message)

        if _tool_name:
            _obs = dispatch_tool(_tool_name, _tool_input)
            _bad_tools_state.messages.append({
                "role": "tool",
                "tool_call_id": _tool_id,
                "content": _obs
            })
            _bad_tools_state.steps.append(
                AgentStep(_step_num, _thought, _tool_name, _tool_input, _obs)
            )
            print(f"Step {_step_num}: {_tool_name}")
            if _tool_name == "finish":
                _bad_tools_state.is_done = True
                _bad_tools_state.final_answer = _tool_input.get("answer", "")
                break
        else:
            print(f"Step {_step_num}: text only")
    except Exception as e:
        print(f"Error: {e}")
        break

print(f"\n{'='*50}")
print(f"Good descriptions -- steps: {len(state_good_tools.steps)}, "
      f"completed: {state_good_tools.is_done}")
print(f"Bad descriptions  -- steps: {len(_bad_tools_state.steps)}, "
      f"completed: {_bad_tools_state.is_done}")

=== Testing with GOOD tool descriptions ===
Step 1: read_file
  Thought: I'll read the file to see its contents and then find the oldest student.
  Observation: name,age,score
Alice,25,92
Bob,30,85
Charlie,22,97
Diana,28,88
Eve,35,91

Step 2: run_python
  Thought: Now I'll write Python code to read the CSV file and find the oldest student:
  Observation: STDOUT:
The oldest student is Eve with age 35

STDERR:

Step 3: run_python
  Thought: The code successfully found the oldest student. Let me verify by checking all ages:
  Observation: STDOUT:
Alice: 25
Bob: 30
Charlie: 22
Diana: 28
Eve: 35

STDERR:

Step 4: finish
  Thought: Perfect! Eve is indeed the oldest student at age 35.
  Observation: TASK COMPLETE: The oldest student is Eve.


=== Testing with BAD tool descriptions ===
Step 1: read_file
Step 2: run_python
Step 3: run_python
Step 4: finish

Good descriptions -- steps: 4, completed: True
Bad descriptions  -- steps: 4, completed: True


### Observations

> **Tool selection accuracy:** With good descriptions, the agent's first tool
> call was ___ . With bad descriptions, it was ___ .
> The agent chose correctly / incorrectly because ___.

> **Connection to production:** Tool descriptions in production APIs (e.g.,
> OpenAI function calling, Anthropic tool use) are essentially ___ for the model.
> They serve the same role as ___ in a traditional software system.

## 11. Extensions and Connections to Production Systems

### What you have built

You now have a working ReAct agent with:
- **Structured tool definitions** that tell the LLM what actions are available
- **A parsing layer** that extracts structured actions from LLM output
- **An execution environment** that runs tools safely with timeouts and error handling
- **Context management** that keeps the conversation within the model's window
- **Loop detection** that catches stuck behavior
- **Error recovery** with retry logic

### How this connects to real systems

| Component you built | Production equivalent |
|---|---|
| `get_tool_definitions()` | Tool/function schemas in Claude API, OpenAI function calling |
| `dispatch_tool()` | MCP (Model Context Protocol) server dispatch |
| `run_python()` | Sandboxed code execution (E2B, Modal, Docker containers) |
| `truncate_context()` | Context window management in Claude Code, Cursor |
| `detect_loop()` | Guardrails and circuit breakers in agent frameworks |
| `get_system_prompt()` | System prompts in every production LLM application |
| `run_production_agent()` | The agent loop in LangChain, CrewAI, Autogen, etc. |

### Extensions to explore (optional)

1. **Add a `write_file` tool**: Let the agent create and save files. This
   requires thinking about security — what directories should it be allowed
   to write to?

2. **Add a `web_search` tool**: Let the agent search the web for documentation.
   How does this change the system prompt?

3. **Multi-agent**: Have two agents collaborate — one writes code, the other
   reviews it. How do you structure the communication?

4. **Streaming**: Instead of waiting for the full response, stream the agent's
   thoughts in real time. The Anthropic API supports streaming — how does this
   change the loop?

5. **Evaluation harness**: Build a suite of coding problems with known answers
   and measure your agent's accuracy across different system prompts and tool
   configurations.

---

### Key takeaways for your ML engineer interviews

1. **The agent loop is a program, not a model.** The LLM is a component inside
   your code. You control when it runs, what it sees, and what it can do.

2. **Prompt engineering is systems engineering.** The system prompt and tool
   descriptions are the "training data" for agent behavior. Bad prompts lead to
   bad agents, regardless of model quality.

3. **Context management is a first-class problem.** Every production agent
   system manages its context window carefully. This is directly analogous to
   memory management in traditional systems programming.

4. **Robustness requires explicit engineering.** Models do not naturally handle
   errors well, avoid loops, or manage resources. All of that is your code.

5. **Tool design is API design.** The tools you give an agent define its
   capabilities. Clear, well-scoped tools with good descriptions are the
   single biggest lever for agent quality.

---

## 12. Summary

| Section | What you implemented | Key concept |
|---|---|---|
| 3 | Tool definitions | Schema design for LLM tool use |
| 4 | Tool implementations | Safe code execution, file reading, dispatching |
| 5 | Agent loop | The ReAct thought-action-observation cycle |
| 7 | Context management | Truncation strategy for finite context windows |
| 8 | Error recovery | Loop detection and retry logic |
| 9 | Production agent | All components integrated |
| 10 | Ablations | System prompt and tool description impact on agent behavior |

**Next project:** Project 6 (CLIP) — You will build a dual-encoder that maps
images and text into a shared embedding space using contrastive learning. The
image encoder will be a ViT (Project 4), the text encoder will be a transformer
(Project 3), and you will learn why contrastive loss with temperature scaling is
the key to making zero-shot classification work.